In [0]:
like_games_path='development_042_silver_sandbox.demand_forecast.like_games_ai_attr'
ts_perf_path='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'# contains the class and, supplier, performance  the games in ts_perf are all found in game_attr_path
sales_path='development_042_silver_sandbox.demand_forecast.sales_silver_comp'# contains sales and lease of games
lease_path='development_042_silver_sandbox.demand_forecast.lease_units_bronz'# only sales

In [0]:

ts_perf_df = spark.table(ts_perf_path)

display(ts_perf_df.select("own_status").distinct())

In [0]:
from pyspark.sql import functions as F

# Load every revenue/ownership type. Each calculation applies its own rules.
like_games_df = spark.table(like_games_path)
ts_perf_df = spark.table(ts_perf_path)
sales_df = spark.table(sales_path)
lease_df = spark.table(lease_path)

display(like_games_df)
display(ts_perf_df)
display(sales_df)
display(lease_df)

In [0]:
%skip
from pyspark.sql import functions as F

sales_friendly_none_df = sales_df.where(
    F.lower(F.trim(F.col("friendly_game_name").cast("string"))) == "none"
)
sales_friendly_null_df = sales_df.where(
    F.col("friendly_game_name").isNull()
)

like_to_sales_none_df = like_games_df.where(
    F.lower(F.trim(F.col("to_sales_name_match").cast("string"))) == "none"
)
like_to_sales_null_df = like_games_df.where(
    F.col("to_sales_name_match").isNull()
)

print(f"sales_df friendly_game_name literal 'none': {sales_friendly_none_df.count()}")
print(f"sales_df friendly_game_name actual null: {sales_friendly_null_df.count()}")
print(f"like_games_df to_sales_name_match literal 'none': {like_to_sales_none_df.count()}")
print(f"like_games_df to_sales_name_match actual null: {like_to_sales_null_df.count()}")

display(sales_friendly_none_df)
display(like_to_sales_none_df)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    StructType,
    StructField,
    StringType,
    DoubleType,
)

# sales_df contains the target games and their two neighbor-list columns.
# Keep a stable row ID while unpacking both lists.
sales_source_df = like_games_df.withColumn(
    "_target_id",
    F.monotonically_increasing_id()
).cache()

# Materialize the IDs so both unpack operations use the same target IDs.
sales_source_df.count()

neighbor_schema = ArrayType(
    StructType([
        StructField("join_game_name_key", StringType(), True),
        StructField("matched_source_name", StringType(), True),
        StructField("similarity_score", DoubleType(), True),
        StructField("distance", DoubleType(), True),
    ])
)

targets_pdf = (
    sales_source_df
    .select("_target_id")
    .distinct()
    .toPandas()
)

def unpack_neighbors(source_df, list_column, source_name):
    # The columns are JSON strings in the CSV-backed table. If they are
    # already parsed arrays, use them directly.
    column_type = dict(source_df.dtypes)[list_column]

    if column_type == "string":
        neighbors_expr = F.from_json(F.col(list_column), neighbor_schema)
    else:
        neighbors_expr = F.col(list_column)

    return (
        source_df
        .withColumn("_neighbors", neighbors_expr)
        .withColumn("_neighbor", F.explode_outer("_neighbors"))
        .select(
            "_target_id",
            F.col("_neighbor.join_game_name_key").alias("neighbor_game"),
            F.col("_neighbor.matched_source_name").alias("matched_source_name"),
            F.col("_neighbor.similarity_score").cast("double").alias("similarity_score"),
            F.col("_neighbor.distance").cast("double").alias("distance"),
        )
        .where(
            F.col("distance").isNotNull()
            & F.col("similarity_score").isNotNull()
        )
        .toPandas()
        .assign(source=source_name)
    )

sales_neighbors_pdf = unpack_neighbors(
    sales_source_df,
    "sales_similar_games",
    "Sales"
)

eilers_neighbors_pdf = unpack_neighbors(
    sales_source_df,
    "eilers_similar_games",
    "Eilers"
)

def median_neighbors_by_cutoff(neighbors_pdf, targets_pdf):
    if neighbors_pdf.empty:
        return pd.DataFrame(columns=["distance_cutoff", "median_neighbors"])

    cutoffs = np.linspace(0, neighbors_pdf["distance"].max(), 101)
    medians = []

    for cutoff in cutoffs:
        counts = (
            neighbors_pdf.loc[neighbors_pdf["distance"] <= cutoff]
            .groupby("_target_id")
            .size()
            .reindex(targets_pdf["_target_id"], fill_value=0)
        )
        medians.append(counts.median())

    return pd.DataFrame({
        "distance_cutoff": cutoffs,
        "median_neighbors": medians,
    })

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for col, (label, neighbors_pdf) in enumerate([
    ("Sales", sales_neighbors_pdf),
    ("Eilers", eilers_neighbors_pdf),
]):
    # Median number of neighbors included at each distance cutoff.
    cutoff_pdf = median_neighbors_by_cutoff(neighbors_pdf, targets_pdf)
    axes[0, col].plot(
        cutoff_pdf["distance_cutoff"],
        cutoff_pdf["median_neighbors"],
        linewidth=2,
    )
    axes[0, col].set_title(f"{label}: Median neighbors by distance cutoff")
    axes[0, col].set_xlabel("Maximum distance to include")
    axes[0, col].set_ylabel("Median neighbors per target")
    axes[0, col].grid(True, alpha=0.3)

    # Distribution of similarity scores for the unpacked neighbor pairs.
    axes[1, col].hist(
        neighbors_pdf["similarity_score"].dropna(),
        bins=25,
        edgecolor="white",
    )
    axes[1, col].set_title(f"{label}: Similarity score distribution")
    axes[1, col].set_xlabel("Similarity score (higher is more similar)")
    axes[1, col].set_ylabel("Number of neighbor pairs")
    axes[1, col].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

sales_source_df.unpersist()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Normalize the game name so capitalization and extra spaces don't
# create separate groups.
deduped = (
    like_games_df
    .withColumn(
        "_dedupe_game_key",
        F.regexp_replace(
            F.lower(F.trim(F.col("join_game_name"))),
            r"\s+",
            " ",
        ),
    )
)

# Score each Eilers neighbor list by its average distance.
# Lower average distance means the list contains closer neighbors.
deduped = (
    deduped
    .withColumn(
        "_valid_eilers_neighbors",
        F.filter(
            F.col("eilers_similar_games"),
            lambda neighbor: neighbor["distance"].isNotNull(),
        ),
    )
    .withColumn(
        "_eilers_avg_distance",
        F.when(
            F.size("_valid_eilers_neighbors") > 0,
            F.aggregate(
                F.col("_valid_eilers_neighbors"),
                F.lit(0.0),
                lambda total, neighbor: (
                    total + neighbor["distance"].cast("double")
                ),
            ) / F.size("_valid_eilers_neighbors"),
        ),
    )
)

dedupe_window = (
    Window
    .partitionBy("_dedupe_game_key", "experiment_id")
    .orderBy(
        F.col("_eilers_avg_distance").asc_nulls_last(),
        F.sha2(
            F.to_json(
                F.struct("eilers_similar_games", "sales_similar_games")
            ),
            256,
        ).asc(),
    )
)

like_games_df= (
    deduped
    .withColumn("_dedupe_rank", F.row_number().over(dedupe_window))
    .where(F.col("_dedupe_rank") == 1)
    .drop(
        "_dedupe_game_key",
        "_valid_eilers_neighbors",
        "_eilers_avg_distance",
        "_dedupe_rank",
    )
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DISTANCE_CUTOFF = 0.40
MIN_NEIGHBORS = 3  # Change this to your chosen minimum.

# The updated notebook 05 supplies separate sales and rental match flags.
required_match_columns = {
    "has_sales_match", "has_rental_match", "has_eilers_ts_match",
    "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match",
}
missing_match_columns = sorted(required_match_columns - set(like_games_df.columns))
if missing_match_columns:
    raise ValueError(f"Run the updated notebook 05 first. Missing: {missing_match_columns}")


def normalized_name(column):
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def valid_match_name(column):
    value = normalized_name(column)
    return value.isNotNull() & ~value.isin("", "none", "null", "nan", "na", "n/a")


def match_flag(column):
    return F.coalesce(normalized_name(column).isin("true", "1", "yes"), F.lit(False))


sales_target_eligible = (
    match_flag(F.col("has_sales_match")) & valid_match_name(F.col("to_sales_name_match"))
)
rental_target_eligible = (
    match_flag(F.col("has_rental_match")) & valid_match_name(F.col("to_rental_name_match"))
)

# One ID per target/experiment; ownership is an additional performance key.
# A target with both commercial matches is kept once here.
like_games_source_df = (
    like_games_df
    .where(match_flag(F.col("has_eilers_ts_match")))
    .where(sales_target_eligible | rental_target_eligible)
    .withColumn("_target_id", F.monotonically_increasing_id())
    .cache()
)

# Materialize IDs before processing either neighbor list.
like_games_source_df.count()

target_info_df = like_games_source_df.select(
    "_target_id",
    F.col("join_game_name").alias("target_game"),
    "game_category",
    "own_status",
    "parent_own_status",
    "game_classification",
    "max_months_since_release",
    "has_sales_match",
    "has_rental_match",
    "has_eilers_ts_match",
    "to_sales_name_match",
    "to_rental_name_match",
    "to_eilers_ts_name_match",
    "experiment_readable_name",
    "experiment_id",
    "distance_method",
    "feature_group_weights",
    'game_release_date',
)


def keep_neighbors_with_minimum(source_df, list_column):
    # The list columns are arrays of structs, as shown in your schema.
    exploded_df = (
        source_df
        .select(
            "_target_id",
            F.explode_outer(F.col(list_column)).alias("_neighbor")
        )
        .select(
            "_target_id",
            F.col("_neighbor.join_game_name_key").alias("neighbor_game"),
            F.col("_neighbor.matched_source_name").alias("matched_source_name"),
            F.col("_neighbor.similarity_score").cast("double").alias("similarity_score"),
            F.col("_neighbor.distance").cast("double").alias("distance"),
        )
        .where(F.col("distance").isNotNull())
    )

    within_counts_df = (
        exploded_df
        .where(F.col("distance") <= DISTANCE_CUTOFF)
        .groupBy("_target_id")
        .agg(F.count("*").alias("neighbors_within_cutoff"))
    )

    # Rank only the candidates beyond the cutoff, nearest first.
    fallback_window = Window.partitionBy("_target_id").orderBy(
        F.col("distance").asc(),
        F.col("neighbor_game").asc_nulls_last(),
    )

    beyond_cutoff_df = (
        exploded_df
        .where(F.col("distance") > DISTANCE_CUTOFF)
        .withColumn("_fallback_rank", F.row_number().over(fallback_window))
        .join(within_counts_df, on="_target_id", how="left")
        .withColumn(
            "_fallback_needed",
            F.greatest(
                F.lit(MIN_NEIGHBORS)
                - F.coalesce(F.col("neighbors_within_cutoff"), F.lit(0)),
                F.lit(0),
            ),
        )
    )

    fallback_neighbors_df = beyond_cutoff_df.where(
        F.col("_fallback_rank") <= F.col("_fallback_needed")
    )

    within_cutoff_df = (
        exploded_df
        .where(F.col("distance") <= DISTANCE_CUTOFF)
        .withColumn("neighbors_within_cutoff", F.lit(None).cast("long"))
        .withColumn("_fallback_rank", F.lit(None).cast("int"))
    )

    retained_df = (
        within_cutoff_df
        .unionByName(fallback_neighbors_df, allowMissingColumns=True)
        .withColumn(
            "retained_within_cutoff",
            F.col("distance") <= DISTANCE_CUTOFF,
        )
        .withColumn(
            "retained_as_min_neighbor_fallback",
            F.col("distance") > DISTANCE_CUTOFF,
        )
        .join(target_info_df, on="_target_id", how="left")
    )

    retained_counts_df = (
        retained_df
        .groupBy("_target_id")
        .agg(F.count("*").alias("neighbors_retained_total"))
    )

    summary_df = (
        target_info_df
        .join(within_counts_df, on="_target_id", how="left")
        .join(retained_counts_df, on="_target_id", how="left")
        .fillna({
            "neighbors_within_cutoff": 0,
            "neighbors_retained_total": 0,
        })
        .withColumn(
            "neighbors_preserved_beyond_cutoff",
            F.col("neighbors_retained_total") - F.col("neighbors_within_cutoff"),
        )
    )

    return retained_df, summary_df


sales_neighbors_kept_df, sales_neighbor_cutoff_summary_df = (
    keep_neighbors_with_minimum(
        like_games_source_df.where(sales_target_eligible), "sales_similar_games"
    )
)

eilers_neighbors_kept_df, eilers_neighbor_cutoff_summary_df = (
    keep_neighbors_with_minimum(like_games_source_df, "eilers_similar_games")
)

display(sales_neighbor_cutoff_summary_df)
display(eilers_neighbor_cutoff_summary_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# The demand curve remains cumulative sales. Apply the sales-only rule here
# because cell 3 now loads the complete sales table, including other revenues.
neighbors = sales_neighbors_kept_df.alias("n")
sales_only_df = sales_df.where(
    normalized_name(F.col("revenue_type")).isin("sale", "sales")
)
sales = sales_only_df.alias("s")

join_condition = (
    F.lower(F.trim(F.col("n.matched_source_name")))
    == F.lower(F.trim(F.col("s.normalized_game_name")))
)
matched_sales_df = (
    neighbors
    .join(sales, join_condition, "inner")
    .select(
        F.col("n._target_id").alias("_target_id"),
        F.col("n.target_game").alias("target_game"),
        F.col("n.neighbor_game").alias("neighbor_game"),
        F.col("n.matched_source_name").alias("matched_source_name"),
        F.col("n.distance").alias("neighbor_distance"),
        F.col("s.revenue_type").alias("revenue_type"),
        F.col("s.beginning_month_date").alias("beginning_month_date"),
        F.col("s.expr1_sum").cast("double").alias("expr1_sum"),
        F.col("n.game_release_date").alias("game_release_date"), 
    )
    .where(
        F.col("beginning_month_date").isNotNull()
        & F.col("expr1_sum").isNotNull()
    )
)

# Find each matched game's first observed sales month, then assign month 1–12.
game_window = Window.partitionBy(
    "_target_id",
    "neighbor_game",
    "matched_source_name",
)

monthly_sales_all_df = (
    matched_sales_df
    .withColumn(
        "_sales_month",
        F.trunc(F.col("beginning_month_date"), "month"),
    )
    .withColumn(
        "_first_sales_month",
        F.min("_sales_month").over(game_window),
    )
    .withColumn(
        "month_index",
        (
            F.floor(
                F.months_between(
                    F.col("_sales_month"),
                    F.col("_first_sales_month"),
                )
            ).cast("int") + 1
        ),
    )
    # Sum duplicate rows within a game, revenue type, and month.
    .groupBy(
        "_target_id",
        "target_game",
        "neighbor_game",
        "matched_source_name",
        "neighbor_distance",
        "revenue_type",
        "month_index",
        "game_release_date",
    )
    .agg(F.sum("expr1_sum").alias("monthly_sales"))
)

monthly_sales_df = monthly_sales_all_df.where(
    F.col("month_index").between(1, 12)
)

# Missing internal sales rows are assumed to mean zero sales.
# Coverage is bounded by actual source dates, never an unobserved future.
# Fill missing months between the first and last observed month with zero,
# so cumulative curves stay aligned by month.
game_keys = [
    "_target_id",
    "target_game",
    "neighbor_game",
    "matched_source_name",
    "neighbor_distance",
    "revenue_type",
    "game_release_date"
]

game_month_ranges_df = (
    monthly_sales_all_df
    .groupBy(*game_keys)
    .agg(F.least(F.max("month_index"), F.lit(12)).alias("_last_month"))
)

month_scaffold_df = (
    game_month_ranges_df
    .withColumn(
        "month_index",
        F.explode(F.sequence(F.lit(1), F.col("_last_month"))),
    )
    .drop("_last_month")
)

aligned_months_df = (
    month_scaffold_df
    .join(
        monthly_sales_df,
        on=game_keys + ["month_index"],
        how="left",
    )
    .withColumn("monthly_sales", F.coalesce(F.col("monthly_sales"), F.lit(0.0)))
)

cumulative_window = (
    Window
    .partitionBy(*game_keys)
    .orderBy("month_index")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

neighbor_cumulative_sales_df = (
    aligned_months_df
    .withColumn(
        "cumulative_sales",
        F.sum("monthly_sales").over(cumulative_window),
    )
)

# One baseline and uncertainty band per target, revenue type, and month.
demand_forecast_curve_df = (
    neighbor_cumulative_sales_df
    .groupBy("_target_id", "target_game", "revenue_type", "month_index","game_release_date")
    .agg(
        F.avg("cumulative_sales").alias("baseline_mean_sales"),
        F.stddev_pop("cumulative_sales").alias("_stddev_sales"),
        F.countDistinct("neighbor_game").alias("neighbor_count"),
    )
    .withColumn("_stddev_sales", F.coalesce(F.col("_stddev_sales"), F.lit(0.0)))
    .withColumn(
        "high_band_2std",
        F.col("baseline_mean_sales") + 2 * F.col("_stddev_sales"),
    )
    .withColumn(
        "low_band_2std",
        F.greatest(
            F.col("baseline_mean_sales") - 2 * F.col("_stddev_sales"),
            F.lit(0.0),
        ),
    )
    .drop("_stddev_sales")
    .orderBy("target_game", "revenue_type", "month_index")
)

display(demand_forecast_curve_df)

In [0]:
# Replace the entire old cell 9 with this cell.
# Run after cells 1-8. Reuses cell 7's target IDs for the performance join.
from pyspark.sql import functions as F

FORECAST_MONTHS = 12
LEASE_VALUE_COLUMN = "source_row_count"
LEASE_METRIC_NAME = "monthly_rental_records"
LEASE_CURVE_LABEL = "Monthly rental records"

if "rental_similar_games" not in like_games_source_df.columns:
    raise ValueError("like_games_df is missing rental_similar_games; run the updated notebook 05 first.")

rental_neighbors_kept_df, rental_neighbor_cutoff_summary_df = (
    keep_neighbors_with_minimum(
        like_games_source_df.where(rental_target_eligible), "rental_similar_games"
    )
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Rental snapshots are monthly levels, not cumulative new placements.
required_lease_columns = {"theme_name_eilers", "month", LEASE_VALUE_COLUMN}
missing = sorted(required_lease_columns - set(lease_df.columns))
if missing:
    raise ValueError(f"Missing lease columns: {missing}")

sales_monthly_source_df = (
    sales_df
    .where(normalized_name(F.col("revenue_type")).isin("sale", "sales"))
    .where(valid_match_name(F.col("normalized_game_name")))
    .select(
        normalized_name(F.col("normalized_game_name")).alias("_source_key"),
        F.lit("sales").alias("revenue_type"),
        F.trunc(F.to_date("beginning_month_date"), "month").alias("_calendar_month"),
        F.col("expr1_sum").cast("double").alias("_monthly_value"),
    )
)
lease_monthly_source_df = (
    lease_df
    .where(valid_match_name(F.col("theme_name_eilers")))
    .select(
        normalized_name(F.col("theme_name_eilers")).alias("_source_key"),
        F.lit("lease").alias("revenue_type"),
        F.trunc(F.to_date("month"), "month").alias("_calendar_month"),
        F.col(LEASE_VALUE_COLUMN).cast("double").alias("_monthly_value"),
    )
)
source_keys = ["revenue_type", "_source_key"]
source_monthly_df = (
    sales_monthly_source_df.unionByName(lease_monthly_source_df)
    .where(F.col("_calendar_month").isNotNull() & F.col("_monthly_value").isNotNull())
    .groupBy(*source_keys, "_calendar_month")
    .agg(F.sum("_monthly_value").alias("_monthly_value"))
    .withColumn("_first_month", F.min("_calendar_month").over(Window.partitionBy(*source_keys)))
    .withColumn("month_index", F.floor(F.months_between("_calendar_month", "_first_month")).cast("int") + 1)
)
source_bounds_df = (
    source_monthly_df.groupBy(*source_keys)
    .agg(
        F.min("_first_month").alias("curve_start_month"),
        F.least(F.max("month_index"), F.lit(FORECAST_MONTHS)).alias("_last_month"),
    )
)
source_fill_window = (
    Window.partitionBy(*source_keys).orderBy("month_index")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
source_curves_df = (
    source_bounds_df
    .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.col("_last_month"))))
    .drop("_last_month")
    .join(source_monthly_df.select(*source_keys, "month_index", "_monthly_value"), on=source_keys + ["month_index"], how="left")
    .withColumn("actual_is_imputed", F.col("_monthly_value").isNull())
    .withColumn(
        "_filled_monthly_value",
        F.when(F.col("revenue_type") == "sales", F.coalesce("_monthly_value", F.lit(0.0)))
        .otherwise(F.last("_monthly_value", ignorenulls=True).over(source_fill_window)),
    )
    .withColumn(
        "source_curve_value",
        F.when(F.col("revenue_type") == "sales", F.sum("_filled_monthly_value").over(source_fill_window))
        .otherwise(F.col("_filled_monthly_value")),
    )
)

# Use the sales list for sales and the rental list for leases.
neighbor_columns = ["_target_id", "neighbor_game", "matched_source_name", "distance"]
demand_neighbors_kept_df = (
    sales_neighbors_kept_df.select(*neighbor_columns).withColumn("revenue_type", F.lit("sales"))
    .unionByName(rental_neighbors_kept_df.select(*neighbor_columns).withColumn("revenue_type", F.lit("lease")))
    .withColumn("_source_key", normalized_name(F.col("matched_source_name")))
    .dropDuplicates(["_target_id", "revenue_type", "_source_key"])
)
neighbor_demand_curves_df = demand_neighbors_kept_df.join(source_curves_df, on=["revenue_type", "_source_key"], how="inner")
demand_baseline_df = (
    neighbor_demand_curves_df.groupBy("_target_id", "revenue_type", "month_index")
    .agg(
        F.avg("source_curve_value").alias("forecast_value"),
        F.stddev_pop("source_curve_value").alias("_stddev"),
        F.countDistinct("_source_key").alias("neighbor_count"),
        F.sum(F.col("actual_is_imputed").cast("long")).alias("filled_neighbor_count"),
    )
    .withColumn("_stddev", F.coalesce("_stddev", F.lit(0.0)))
    .withColumn("high_band_2std", F.col("forecast_value") + 2 * F.col("_stddev"))
    .withColumn("low_band_2std", F.greatest(F.col("forecast_value") - 2 * F.col("_stddev"), F.lit(0.0)))
    .drop("_stddev")
)
display(demand_baseline_df.groupBy("revenue_type").agg(F.countDistinct("_target_id").alias("games")))


from pyspark.sql import functions as F

# A shared game ID can have a sales series, a lease series, or both.
target_channel_info_df = (
    like_games_source_df
    .withColumnRenamed("own_status", "target_own_status")
    .withColumn("revenue_type", F.explode(F.array(
        F.when(sales_target_eligible, F.lit("sales")),
        F.when(rental_target_eligible, F.lit("lease")),
    )))
    .where(F.col("revenue_type").isNotNull())
    .select(
        "_target_id", F.col("_target_id").alias("game_id"),
        F.col("join_game_name").alias("target_game"), "revenue_type", "target_own_status",
        "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match",
        "has_sales_match", "has_rental_match", "has_eilers_ts_match",
        "experiment_id", F.col("experiment_readable_name").alias("experiment_name"),
        "distance_method", "feature_group_weights", "game_release_date",
    )
    .withColumn("own_status", F.when(F.col("revenue_type") == "sales", "owned").otherwise("leased"))
    .withColumn("metric_name", F.when(F.col("revenue_type") == "sales", "cumulative_sales").otherwise(LEASE_METRIC_NAME))
    .withColumn("curve_label", F.when(F.col("revenue_type") == "sales", "Cumulative sales").otherwise(LEASE_CURVE_LABEL))
    .withColumn("curve_kind", F.when(F.col("revenue_type") == "sales", "cumulative").otherwise("monthly"))
    .withColumn("_source_key", normalized_name(F.when(
        F.col("revenue_type") == "sales", F.col("to_sales_name_match")
    ).otherwise(F.col("to_rental_name_match"))))
)

# Preserve targets when only actuals or only neighbor forecasts are present.
# Observations and their fills stop at the last supported source month.
demand_forecast_curve_df = (
    target_channel_info_df
    .join(source_bounds_df.select(*source_keys, "curve_start_month"), on=source_keys, how="left")
    .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(FORECAST_MONTHS))))
    .join(demand_baseline_df, on=["_target_id", "revenue_type", "month_index"], how="left")
    .join(
        source_curves_df.select(*source_keys, "month_index", F.col("source_curve_value").alias("actual_value"), "actual_is_imputed"),
        on=source_keys + ["month_index"], how="left",
    )
    .withColumn("forecast_month", F.add_months("curve_start_month", F.col("month_index") - 1))
    # Keep legacy sales columns sales-only for existing consumers.
    .withColumn("baseline_mean_sales", F.when(F.col("revenue_type") == "sales", F.col("forecast_value")))
    .withColumn("actual_cumulative_sales", F.when(F.col("revenue_type") == "sales", F.col("actual_value")))
    .withColumn("actual_sales_month_zero_filled", F.when(F.col("revenue_type") == "sales", F.col("actual_is_imputed")))
    .withColumn("baseline_mean_lease", F.when(F.col("revenue_type") == "lease", F.col("forecast_value")))
    .withColumn("actual_monthly_lease", F.when(F.col("revenue_type") == "lease", F.col("actual_value")))
    .drop("_source_key")
    .orderBy("target_game", "experiment_id", "revenue_type", "month_index")
)
display(demand_forecast_curve_df)


In [0]:
display(demand_forecast_curve_df.select("target_game").distinct())

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

performance = [
    "avg_coin_in_index_vs_house",
    "avg_theo_net_win_index_vs_house",
]

def yearmonth_to_date(column):
    return F.trunc(F.to_date(column.cast("string"), "yyyy-MM-dd"), "month")


# Only owned and leased Eilers rows participate. Null/blank/other statuses
# are excluded here, and cannot enter forecasts or actuals in cell 12.
performance_source_df = (
    ts_perf_df
    .withColumn("own_status", normalized_name(F.col("own_status")))
    .where(F.col("own_status").isin("owned", "leased"))
    .withColumn("_game_key", normalized_name(F.col("game_name")))
    .withColumn("_perf_month", yearmonth_to_date(F.col("yearmonth")))
    .withColumn("no_of_slots", F.col("no_of_slots").cast("double"))
    .where(
        valid_match_name(F.col("game_name"))
        & (F.col("no_of_slots") > 0)
        & F.col("_perf_month").isNotNull()
    )
)
for metric in performance:
    performance_source_df = performance_source_df.withColumn(metric, F.col(metric).cast("double"))

# Sales match -> owned; rental match -> leased. Both matches allow both.
# own_status in the output describes the performance population. Preserve
# the upstream target descriptor separately, since a game can have both.
target_base_df = (
    like_games_source_df
    .withColumnRenamed("own_status", "target_own_status")
    .withColumn(
        "own_status",
        F.explode(F.array(
            F.when(sales_target_eligible, F.lit("owned")),
            F.when(rental_target_eligible, F.lit("leased")),
        )),
    )
    .where(F.col("own_status").isNotNull())
    .select(
        "_target_id",
        F.col("join_game_name").alias("target_game"),
        "own_status", "target_own_status",
        "has_sales_match", "has_rental_match", "has_eilers_ts_match",
        "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match",
        "experiment_id", "game_release_date", "distance_method", "feature_group_weights",
        F.col("experiment_readable_name").alias("experiment_name"),
    )
    .withColumn(
        "revenue_type",
        F.when(F.col("own_status") == "owned", F.lit("sales")).otherwise(F.lit("lease")),
    )
    .withColumn(
        "_game_key",
        normalized_name(F.when(
            valid_match_name(F.col("to_eilers_ts_name_match")), F.col("to_eilers_ts_name_match")
        ).otherwise(F.col("target_game"))),
    )
)

# Require a target Eilers entry for that ownership type and anchor its
# curve to the first observed month of that same type.
target_start_df = (
    performance_source_df
    .groupBy("_game_key", "own_status")
    .agg(F.min("_perf_month").alias("_target_start_month"))
)
performance_targets_df = target_base_df.join(
    target_start_df, on=["_game_key", "own_status"], how="inner"
)

# Continue to use the Eilers neighbor list for performance. Commercial
# matches gate the targets; ownership selects each neighbor's Eilers rows.
neighbors = (
    eilers_neighbors_kept_df.drop("own_status")
    .join(performance_targets_df.select("_target_id", "own_status"), on="_target_id", how="inner")
    .alias("n")
)
perf = performance_source_df.alias("p")

neighbor_perf_df = (
    neighbors
    .join(
        perf,
        (normalized_name(F.col("n.matched_source_name")) == F.col("p._game_key"))
        & (F.col("n.own_status") == F.col("p.own_status")),
        "inner",
    )
    .select(
        F.col("n._target_id"),
        F.col("n.target_game"),
        F.col("n.own_status"),
        F.col("n.matched_source_name"),
        F.col("p.no_of_slots"),
        F.col("p._perf_month"),
        *[F.col(f"p.{metric}").alias(metric) for metric in performance],
    )
)

# Align each neighbor's observations to its own first observed performance month.
neighbor_window = Window.partitionBy(
    "_target_id",
    "own_status",
    "matched_source_name",
)

neighbor_perf_df = (
    neighbor_perf_df
    .withColumn("_neighbor_start_month", F.min("_perf_month").over(neighbor_window))
    .withColumn(
        "month_index",
        F.floor(
            F.months_between("_perf_month", "_neighbor_start_month")
        ).cast("int") + 1,
    )
)

# Make one row per target, neighbor, slot count, and relative month.
neighbor_monthly_df = (
    neighbor_perf_df
    .where(F.col("month_index").between(1, 12))
    .groupBy(
        "_target_id",
        "target_game",
        "own_status",
        "matched_source_name",
        "no_of_slots",
        "month_index",
    )
    .agg(*[F.avg(metric).alias(metric) for metric in performance])
)

# Calculate the no_of_slots-weighted mean and population standard deviation.
group_cols = ["_target_id", "target_game", "own_status", "month_index"]
aggregations = []

for metric in performance:
    valid_weight = F.when(F.col(metric).isNotNull(), F.col("no_of_slots"))
    aggregations.extend([
        F.sum(valid_weight).alias(f"_{metric}_weight_sum"),
        F.sum(valid_weight * F.col(metric)).alias(f"_{metric}_weighted_sum"),
        F.sum(valid_weight * F.pow(F.col(metric), 2))
            .alias(f"_{metric}_weighted_sq_sum"),
    ])

# Consolidate each metric's weighted statistics to one game-month.
neighbor_game_keys = ["_target_id", "target_game", "own_status", "matched_source_name"]
neighbor_monthly_stats_df = (
    neighbor_monthly_df
    .groupBy(*neighbor_game_keys, "month_index")
    .agg(*aggregations)
)

# Use all source dates to establish the last supported month, then cap at 12.
# Do not create months beyond a game's last observed performance date.
neighbor_month_ranges_df = (
    neighbor_perf_df
    .groupBy(*neighbor_game_keys)
    .agg(F.least(F.max("month_index"), F.lit(12)).alias("_last_month"))
)
neighbor_month_scaffold_df = (
    neighbor_month_ranges_df
    .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.col("_last_month"))))
    .drop("_last_month")
)
neighbor_aligned_stats_df = neighbor_month_scaffold_df.join(
    neighbor_monthly_stats_df,
    on=neighbor_game_keys + ["month_index"],
    how="left",
)
neighbor_fill_window = (
    Window.partitionBy(*neighbor_game_keys)
    .orderBy("month_index")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

filled_aggregations = []
for metric in performance:
    stat_columns = [
        f"_{metric}_weight_sum",
        f"_{metric}_weighted_sum",
        f"_{metric}_weighted_sq_sum",
    ]
    observed_col = f"{metric}_is_observed"
    filled_col = f"{metric}_is_filled"
    valid_stats = F.when(
        F.col(stat_columns[0]) > 0,
        F.struct(*[F.col(name).alias(name) for name in stat_columns]),
    )
    neighbor_aligned_stats_df = (
        neighbor_aligned_stats_df
        .withColumn(observed_col, F.coalesce(F.col(stat_columns[0]) > 0, F.lit(False)))
        .withColumn("_last_metric_stats", F.last(valid_stats, ignorenulls=True).over(neighbor_fill_window))
        .withColumn(filled_col, (~F.col(observed_col)) & F.col("_last_metric_stats").isNotNull())
    )
    for name in stat_columns:
        neighbor_aligned_stats_df = neighbor_aligned_stats_df.withColumn(
            name, F.col("_last_metric_stats").getField(name)
        )
    neighbor_aligned_stats_df = neighbor_aligned_stats_df.drop("_last_metric_stats")

    # Leading nulls stay null. Filled observations do not count as observed.
    filled_aggregations.extend([
        *[F.sum(name).alias(name) for name in stat_columns],
        F.count(F.when(F.col(stat_columns[0]) > 0, F.lit(1)))
            .alias(f"{metric}_neighbor_count"),
        F.sum(F.col(observed_col).cast("long"))
            .alias(f"{metric}_observed_neighbor_count"),
        F.sum(F.col(filled_col).cast("long"))
            .alias(f"{metric}_filled_neighbor_count"),
    ])

performance_by_month_df = (
    neighbor_aligned_stats_df
    .groupBy(*group_cols)
    .agg(*filled_aggregations)
)

for metric in performance:
    weight_sum = F.col(f"_{metric}_weight_sum")
    mean_col = f"{metric}_mean"
    std_col = f"{metric}_std"

    performance_by_month_df = (
        performance_by_month_df
        .withColumn(
            mean_col,
            F.when(
                weight_sum > 0,
                F.col(f"_{metric}_weighted_sum") / weight_sum,
            ),
        )
        .withColumn(
            std_col,
            F.when(
                weight_sum > 0,
                F.sqrt(
                    F.greatest(
                        F.col(f"_{metric}_weighted_sq_sum") / weight_sum
                        - F.pow(F.col(mean_col), 2),
                        F.lit(0.0),
                    )
                ),
            ),
        )
        .withColumn(f"{metric}_lower_2std", F.col(mean_col) - 2 * F.col(std_col))
        .withColumn(f"{metric}_higher_2std", F.col(mean_col) + 2 * F.col(std_col))
    )

# Each target/ownership type gets its own 12-month calendar.
target_months_df = (
    performance_targets_df
    .withColumn("month_index", F.explode(F.sequence(F.lit(1), F.lit(12))))
    .withColumn(
        "forecast_yearmonth",
        F.date_format(
            F.add_months("_target_start_month", F.col("month_index") - 1),
            "yyyyMM",
        ),
    )
    .drop("_game_key", "_target_start_month")
)

performance_forecast_df = (
    target_months_df
    .join(
        performance_by_month_df,
        on=["_target_id", "target_game", "own_status", "month_index"],
        how="left",
    )
    .orderBy("target_game", "own_status", "month_index")
)

display(performance_forecast_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

DEMAND_TABLE = (
    "development_042_silver_sandbox.demand_forecast.demand_forecast_results"
)
PERFORMANCE_TABLE = (
    "development_042_silver_sandbox.demand_forecast.performance_forecast_results"
)

# Check the in-memory demand output before either table is overwritten.
required_demand_columns = {
    "_target_id", "target_game", "revenue_type", "month_index", "game_release_date",
    "forecast_value", "actual_value", "actual_is_imputed", "metric_name",
    "curve_label", "own_status", "experiment_id", "experiment_name",
    "to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match",
    "low_band_2std", "high_band_2std",
}
missing_before_save = sorted(required_demand_columns - set(demand_forecast_curve_df.columns))
if missing_before_save:
    raise ValueError(
        "The old demand DataFrame is still in memory. Run the replacement cell 9 "
        f"in this notebook before saving. Missing columns: {missing_before_save}"
    )

# Use exactly the same valid Eilers ownership rows as the forecasts.
actual_perf = performance_source_df.withColumnRenamed("_perf_month", "_actual_month")

actual_perf_aggs = []
for metric in performance:
    valid_weight = F.when(F.col(metric).isNotNull(), F.col("no_of_slots"))
    actual_perf_aggs.append(
        (
            F.sum(valid_weight * F.col(metric))
            / F.sum(valid_weight)
        ).alias(f"{metric}_actual")
    )

actual_perf_monthly_df = (
    actual_perf
    .groupBy("_game_key", "own_status", "_actual_month")
    .agg(*actual_perf_aggs)
)

# Add actual performance to the performance forecast by target and month.
f = performance_forecast_df.alias("f")
a = actual_perf_monthly_df.alias("a")

forecast_month = F.to_date(
    F.concat(F.col("f.forecast_yearmonth").cast("string"), F.lit("01")),
    "yyyyMMdd",
)

performance_forecast_with_actuals_df = (
    f.join(
        a,
        (
            normalized_name(F.when(
                valid_match_name(F.col("f.to_eilers_ts_name_match")), F.col("f.to_eilers_ts_name_match")
            ).otherwise(F.col("f.target_game"))) == F.col("a._game_key")
        )
        & (F.col("f.own_status") == F.col("a.own_status"))
        & (forecast_month == F.col("a._actual_month")),
        "left",
    )
    .select(
        "f.*",
        *[F.col(f"a.{metric}_actual") for metric in performance],
    )
)

# Filled display curves are separate from observed actuals used for errors.
# Using each game's full source range recognizes gaps even when the next
# recorded month falls beyond the 12-month plotting horizon.
actual_perf_bounds_df = (
    actual_perf
    .groupBy("_game_key", "own_status")
    .agg(F.max("_actual_month").alias("_actual_perf_last_month"))
)
performance_forecast_with_actuals_df = (
    performance_forecast_with_actuals_df
    .withColumn(
        "_game_key",
        normalized_name(F.when(
            valid_match_name(F.col("to_eilers_ts_name_match")), F.col("to_eilers_ts_name_match")
        ).otherwise(F.col("target_game"))),
    )
    .join(actual_perf_bounds_df, on=["_game_key", "own_status"], how="left")
    .withColumn(
        "_forecast_month",
        F.to_date(F.concat(F.col("forecast_yearmonth").cast("string"), F.lit("01")), "yyyyMMdd"),
    )
)
actual_perf_fill_window = (
    Window.partitionBy("_target_id", "own_status")
    .orderBy("month_index")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
for metric in performance:
    actual_col = f"{metric}_actual"
    filled_col = f"{metric}_actual_filled"
    performance_forecast_with_actuals_df = (
        performance_forecast_with_actuals_df
        .withColumn(
            filled_col,
            F.when(
                F.col("_forecast_month") <= F.col("_actual_perf_last_month"),
                F.last(F.col(actual_col), ignorenulls=True).over(actual_perf_fill_window),
            ),
        )
        .withColumn(
            f"{metric}_actual_is_filled",
            F.col(actual_col).isNull() & F.col(filled_col).isNotNull(),
        )
    )
performance_forecast_with_actuals_df = performance_forecast_with_actuals_df.drop(
    "_game_key", "_actual_perf_last_month", "_forecast_month"
)

# Save the demand and performance forecasts.
(
    demand_forecast_curve_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DEMAND_TABLE)
)

(
    performance_forecast_with_actuals_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PERFORMANCE_TABLE)
)

# Verify the persisted table, so notebook 07 never depends on an old DataFrame.
#spark.catalog.refreshTable(DEMAND_TABLE)
#spark.catalog.refreshTable(PERFORMANCE_TABLE)
saved_demand_df = spark.table(DEMAND_TABLE)
missing_after_save = sorted(required_demand_columns - set(saved_demand_df.columns))
if missing_after_save:
    raise RuntimeError(f"Saved demand table is missing required columns: {missing_after_save}")

print("DEMAND_OUTPUT_READY_FOR_07")
print(f"Verified table: {DEMAND_TABLE}")
print(f"Saved columns: {saved_demand_df.columns}")
display(
    saved_demand_df.groupBy("revenue_type", "own_status")
    .agg(
        F.countDistinct("_target_id").alias("target_games"),
        F.count("forecast_value").alias("forecast_rows"),
        F.count("actual_value").alias("actual_rows"),
    )
)
print("Now rerun notebook 07 cell 3 to reload the saved tables, then cell 5.")
